# Provider institution type per country

**Research question:** What kind of institutions contribute to Europeana on behalf of each country? The purpose is to explore whether a country's Europeana profile is shaped by the kind of institutions that contribute to it.

### Categories

Categories are not an official Europeana taxonomy — they were defined for this analysis, built iteratively from the institution names and Wikidata descriptions actually observed in
the data (not decided upfront):

- `library/archive`
- `art/history museum`
- `natural history/science institution`
- `audiovisual/film archive`
- `academic/research institution`
- `administrative body`
- `other` — providers that don't fit any category aboveh

This step explores what is the most field contributing and pushing on the digitization for each country, it permits to understand whether a country's Europeana profile is shaped by **what kind of institutions** contribute.

**Process:**
1. Total item count per country - via the Europeana Search API
2. Full `dataProvider` list per country - every distinct provider and its item count, via facet
3. Resolve each provider on Wikidata - English search first; if that fails or lacks a description, retry in the provider's own country language; translate non-English descriptions to Englihs. A SPARQL fallback checks `instanceOf` (P31) and `fieldOfWork`(P101) for providers the initial search didn't resolve well.
4. Classify - using both the Wikidata description and the insitution's own name, since descriptions don't always restate what the name already makes clear. Rules cover several languages for the most common institution-type terms - built by reviewing real unmatched results, not guessed in advance.
5. Review and manually correct - `other`and unresolved results, including cases where Wikidata's fuzzy search returned a wrong entity entirely.
6. Aggregate and visualize - item-volume-weighted category share per country.

## 1. Setup

In [1]:
import os
import time
import json
import re
from pathlib import Path

import requests
import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv
from deep_translator import GoogleTranslator

load_dotenv()       # reads a local .env file with EUROPEANA_API_KEY=xxxxxxxx
API_KEY = os.environ["EUROPEANA_API_KEY"]

BASE_URL = "https://api.europeana.eu/record/v2/search.json"
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COUNTRIES = [
    "italy",
    "france",
    "germany",
    "spain",
    "netherlands",
    "portugal",
]

LANG_BY_COUNTRY = {
    "italy": "it", "france": "fr", "germany": "de",
    "spain": "es", "netherlands": "nl", "portugal": "pt",
}

REQUEST_DELAY = 0.3  # seconds between calls, be polite to the API

## 2. Europeana request helper

In [2]:
def europeana_search(query="*", qf=None, facet=None, rows=0, **extra):
    params = {"wskey": API_KEY, "query": query, "rows": rows, **extra}
    if qf:
        params["qf"] = qf
    if facet:
        params["facet"] = facet
        params["profile"] = "facets"

    for attempt in range(5):
        resp = requests.get(BASE_URL, params=params)
        if resp.status_code == 429:
            wait = 2 ** attempt
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        if resp.status_code != 200:
            print("URL:", resp.url)
            print("Status:", resp.status_code)
            print("Body:", resp.text[:1000])
            resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return resp.json()

    raise RuntimeError(f"Failed after retries: {params}")


test = europeana_search(query="*", qf=["COUNTRY:italy"], rows=1)
print("success:", test.get("success"), "| totalResults:", test.get("totalResults"))


success: True | totalResults: 1832376


## 3. Total items per country

In [3]:
def get_total_items(country):
    result = europeana_search(query="*", qf=[f"COUNTRY:{country}"], rows=0)
    return {"country": country, "total_items": result.get("totalResults", 0)}


totals_df = pd.DataFrame([get_total_items(c) for c in TARGET_COUNTRIES])
totals_df.to_csv(DATA_DIR / "total_items_by_country.csv", index=False)
totals_df

,country,total_items
0,italy,1832376
1,france,4724898
2,germany,8701240
3,spain,6581724
4,netherlands,9204845
5,portugal,139858


## 4. Full provider list per country

In [4]:
def get_all_providers(country, page_size=200, max_pages=50):
    rows = []
    offset = 0
    for _ in range(max_pages):
        result = europeana_search(
            query="*",
            qf=[f"COUNTRY:{country}"],
            facet="DATA_PROVIDER",
            rows=0,
            **{"f.DATA_PROVIDER.facet.limit": page_size, "f.DATA_PROVIDER.facet.offset": offset},
        )
        facets = result.get("facets", [])
        page_rows = [
            {"country": country, "provider": f["label"], "count": f["count"]}
            for facet in facets
            for f in facet.get("fields", [])
        ]
        rows.extend(page_rows)
        if len(page_rows) < page_size:
            break
        offset += page_size
    return rows


providers_by_country = {}
target_folder = DATA_DIR / "providers_data"
target_folder.mkdir(parents=True, exist_ok=True)

for country in TARGET_COUNTRIES:
    rows = get_all_providers(country)
    country_df = pd.DataFrame(rows)
    country_df.to_csv(target_folder / f"providers_{country}.csv", index=False)
    providers_by_country[country] = country_df
    print(f"{country}: {len(country_df)} distinct providers -> providers_{country}.csv")

providers_df = pd.concat(providers_by_country.values(), ignore_index=True)

italy: 158 distinct providers -> providers_italy.csv
france: 50 distinct providers -> providers_france.csv
germany: 375 distinct providers -> providers_germany.csv
spain: 275 distinct providers -> providers_spain.csv
netherlands: 104 distinct providers -> providers_netherlands.csv
portugal: 40 distinct providers -> providers_portugal.csv


In [5]:
# for each country, what % of total item volume the top N providers represent - how large N needs to be.

def coverage_at_n(country_df, n):
    sorted_df = country_df.sort_values("count", ascending=False)
    return sorted_df["count"].head(n).sum() / sorted_df["count"].sum()

for country, df in providers_by_country.items():
    print(f"{country}: top 20 -> {coverage_at_n(df, 20):.1%}, top 50 -> {coverage_at_n(df, 50):.1%}, "
          f"top 100 -> {coverage_at_n(df, 100):.1%} (of {len(df)} total providers)")


italy: top 20 -> 90.2%, top 50 -> 98.4%, top 100 -> 99.9% (of 158 total providers)
france: top 20 -> 99.7%, top 50 -> 100.0%, top 100 -> 100.0% (of 50 total providers)
germany: top 20 -> 79.6%, top 50 -> 92.2%, top 100 -> 97.7% (of 375 total providers)
spain: top 20 -> 76.7%, top 50 -> 92.3%, top 100 -> 98.4% (of 275 total providers)
netherlands: top 20 -> 89.8%, top 50 -> 99.5%, top 100 -> 100.0% (of 104 total providers)
portugal: top 20 -> 99.7%, top 50 -> 100.0%, top 100 -> 100.0% (of 40 total providers)


In [6]:
# cut each country's provider list down to its top 100 by item count

TOP_N_PER_COUNTRY = 100

top_providers_rows = []
for country, df in providers_by_country.items():
    top = df.sort_values("count", ascending=False).head(TOP_N_PER_COUNTRY)
    top_providers_rows.append(top)

top_providers_df = pd.concat(top_providers_rows, ignore_index=True)
distinct_providers = top_providers_df[["provider", "country"]].drop_duplicates()
print(f"{len(top_providers_df)} (country, provider) rows / {len(distinct_providers)} distinct providers selected")

490 (country, provider) rows / 490 distinct providers selected


## 5. Wikidata pipeline

In [7]:
WD_SEARCH_URL = "https://www.wikidata.org/w/api.php"
WD_HEADERS = {
    "User-Agent": "InfoVis-course-project/0.1 (student project; contact: your.email@example.com)"
}
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"


def search_wikidata_rest(name, lang="en", max_retries=3):
    params = {
        "action": "wbsearchentities", "search": name, "language": lang,
        "format": "json", "limit": 1, "type": "item",
    }
    for attempt in range(max_retries):
        try:
            resp = requests.get(WD_SEARCH_URL, params=params, headers=WD_HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"network error searching '{name}': {e}")
            time.sleep(2 ** attempt)
            continue
        if resp.status_code == 200:
            results = resp.json().get("search", [])
            if not results:
                return {"qid": None, "description": None, "desc_lang": None, "status": "no_match"}
            top = results[0]
            display_desc = top.get("display", {}).get("description", {})
            description = display_desc.get("value") or top.get("description")
            actual_lang = display_desc.get("language")
            return {"qid": top["id"], "description": description,
                    "desc_lang": actual_lang if description else None, "status": "ok"}
        if resp.status_code in (429, 502, 503):
            time.sleep(2 ** attempt)
            continue
        return {"qid": None, "description": None, "desc_lang": None, "status": f"http_{resp.status_code}"}
    return {"qid": None, "description": None, "desc_lang": None, "status": "retries_exhausted"}


def resolve_provider_wikidata(name, country):
    result = search_wikidata_rest(name, lang="en")
    if result["status"] == "ok" and result["description"]:
        result["resolution_lang"] = "en"
        return result
    country_lang = LANG_BY_COUNTRY.get(country, "en")
    local_result = search_wikidata_rest(name, lang=country_lang)
    if local_result["status"] == "ok" and (local_result["description"] or not result["qid"]):
        local_result["resolution_lang"] = country_lang
        return local_result
    if result["status"] == "ok":
        result["resolution_lang"] = "en"
        return result
    local_result["resolution_lang"] = country_lang
    return local_result


wikidata_resolution = {}
for _, row in distinct_providers.iterrows():
    name, country = row["provider"], row["country"]
    result = resolve_provider_wikidata(name, country)
    wikidata_resolution[name] = result
    time.sleep(0.3)

wd_resolved = sum(1 for r in wikidata_resolution.values() if r["status"] == "ok")
print(f"{wd_resolved} / {len(wikidata_resolution)} resolved via Wikidata")

185 / 490 resolved via Wikidata


### 6.1 Translate non-English descriptions

In [8]:
translation_cache = {}


def translate_to_english(text, src_lang):
    if not text:
        return None
    if src_lang == "en":
        return text
    key = (src_lang, text)
    if key in translation_cache:
        return translation_cache[key]
    try:
        translated = GoogleTranslator(source=src_lang, target="en").translate(text)
    except Exception as e:
        print(f"translation failed for lang={src_lang}: {e}")
        translated = None
    translation_cache[key] = translated
    time.sleep(0.2)
    return translated


for name, info in wikidata_resolution.items():
    info["description_en"] = translate_to_english(info.get("description"), info.get("desc_lang"))


### 6.2 SPARQL fallback looking for `instance_of`/`field_of_work`

In [9]:
def fetch_institution_info(qids, batch_size=50):
    results = {}
    qids = [q for q in qids if q]
    for i in range(0, len(qids), batch_size):
        batch = qids[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in batch)
        query = f"""
        SELECT ?item ?instanceOfLabel ?fieldOfWorkLabel WHERE {{
          VALUES ?item {{ {values} }}
          OPTIONAL {{ ?item wdt:P31 ?instanceOf . ?instanceOf rdfs:label ?instanceOfLabel . FILTER(LANG(?instanceOfLabel) = "en") }}
          OPTIONAL {{ ?item wdt:P101 ?fieldOfWork . ?fieldOfWork rdfs:label ?fieldOfWorkLabel . FILTER(LANG(?fieldOfWorkLabel) = "en") }}
        }}
        """
        resp = requests.get(WIKIDATA_SPARQL, params={"query": query, "format": "json"}, headers=WD_HEADERS)
        if resp.status_code != 200:
            continue
        for row in resp.json()["results"]["bindings"]:
            qid = row["item"]["value"].rsplit("/", 1)[-1]
            entry = results.setdefault(qid, {"instance_of": [], "field_of_work": []})
            if "instanceOfLabel" in row:
                entry["instance_of"].append(row["instanceOfLabel"]["value"])
            if "fieldOfWorkLabel" in row:
                entry["field_of_work"].append(row["fieldOfWorkLabel"]["value"])
        time.sleep(0.5)
    return results


qid_list = [info["qid"] for info in wikidata_resolution.values() if info.get("qid")]
sparql_info = fetch_institution_info(qid_list)
print(f"instance_of/field_of_work found for {len(sparql_info)} / {len(qid_list)} QIDs")


instance_of/field_of_work found for 185 / 185 QIDs


## 6. Classify

Checks the Wikidata description first, then the institution's own name, then the SPARQL `instance_of`/`field_of_work` fallback. Rules cover several languages - built from reviewing real unmatched results.


In [10]:
INSTITUTION_RULES = [
    (r"film archive|cin[ée]math[èe]que|kinemathek|audiovisual archive|\baudiovisual\b|\bcinema\b|cinecitt[aà]|"
     r"moving image|sound\s*(&|and)\s*vision|film museum|film institute", "audiovisual/film archive"),
    (r"\bmusic\b|musical|ethnomusicology|conservator(y|io|oire)|philharmonic|philharmonie|concert hall",
     "audiovisual/film archive"),

    (r"national library|public library|research library|\blibrar", "library/archive"),
    (r"\barchiv|\barquiv|national archives|repositor", "library/archive"),
    (r"bibliotec|bibliothek", "library/archive"),

    (r"natural history museum|science museum|herbarium|botanic|\bzoo\b|planetarium|aquarium|"
     r"geological survey|tropical.{0,15}research", "natural history/science institution"),

    (r"art museum|history museum|national museum|encyclopedic museum|archaeological museum|"
     r"specialized museum|military museum|open-air museum|\bmuseum\b|\bmuseo\b|mus[ée]e|\bmuseu\b|"
     r"museen|\bgallery\b|art collection|photograph|\bcastle\b|\bpalace\b|historical monument|"
     r"epigraphic|archaeolog|archeolog", "art/history museum"),

    (r"broadcaster|television|radio station|newspaper|press agency|publishing house",
     "media/broadcast organization"),

    (r"\buniversity\b|research institute|academy of sciences|research cent(er|re)|\bresearch\b|"
     r"\blaboratory\b|laboratoire", "academic/research institution"),

    (r"government agency|ministry|municipality|public administration|state institution|\bgovernment\b|"
     r"gobierno|ajuntament|gemeente|province of|heritage agency|superintendence|\bcourt\b",
     "government/administrative body"),
]
_compiled_institution_rules = [(re.compile(pat, re.IGNORECASE), name) for pat, name in INSTITUTION_RULES]


def classify_text(text):
    if not text:
        return None
    for pattern, category in _compiled_institution_rules:
        if pattern.search(text):
            return category
    return "other"


def classify_from_sparql(qid):
    info = sparql_info.get(qid, {"instance_of": [], "field_of_work": []})

    fow_category = classify_text(" | ".join(info["field_of_work"]))
    if fow_category not in (None, "other"):
        return fow_category

    io_category = classify_text(" | ".join(info["instance_of"]))
    if io_category not in (None, "other"):
        return io_category

    if not info["field_of_work"] and not info["instance_of"]:
        return None
    return "other"


def classify_provider(name):
    name_category = classify_text(name)
    wd_info = wikidata_resolution.get(name)
    desc_category = classify_text(wd_info.get("description_en")) if wd_info else None

    if name_category not in (None, "other") and desc_category not in (None, "other") and name_category != desc_category:
        # disagreement -- flag for review rather than silently picking one
        return "other", "name_description_conflict"
    if name_category not in (None, "other"):
        return name_category, "provider_name"
    if desc_category not in (None, "other"):
        return desc_category, "wikidata_description"

    if wd_info:
        sparql_category = classify_from_sparql(wd_info.get("qid"))
        if sparql_category not in (None, "other"):
            return sparql_category, "wikidata_sparql_fallback"
        if desc_category == "other" or sparql_category == "other" or name_category == "other":
            return "other", "unmatched"

    if name_category == "other":
        return "other", "unmatched"

    return None, None

classification_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider(name)
    classification_rows.append({"provider": name, "provider_category": category, "classification_source": source})

classification_df = pd.DataFrame(classification_rows)
classification_df.to_csv(DATA_DIR / "provider_classification.csv", index=False)
print(classification_df["provider_category"].value_counts(dropna=False))
print()
print(classification_df["classification_source"].value_counts(dropna=False))


provider_category
library/archive                        201
art/history museum                     139
other                                   78
academic/research institution           29
audiovisual/film archive                17
government/administrative body          14
natural history/science institution      9
media/broadcast organization             3
Name: count, dtype: int64

classification_source
provider_name                373
unmatched                     71
wikidata_description          29
wikidata_sparql_fallback      10
name_description_conflict      7
Name: count, dtype: int64


## 7. Confirmed fixes (hard-coded, independent of manual_review.csv)

These are specific misclassifications identified and confirmed during review -- kept in code,
not only in the CSV, so they can never be silently lost due to a stale reload or an
out-of-order cell run (as happened before). `classify_provider_final` checks this dict first,
before the CSV-based `manual_overrides`.


In [11]:
CONFIRMED_FIXES = {
    # Entity-resolution errors (Wikidata matched the wrong entity or a misleading description)
    "Brixiana": "library/archive",  # confirmed via the platform's own description -- a digital library
    "Paul Van Riel": "other",  # individual photographer, not an institution

    # Government heritage-protection agencies mismatched by museum/archaeology keywords
    "Cultural Heritage Agency of the Netherlands": "government/administrative body",
    "Historical Monuments: Regional Conservation": "government/administrative body",
    "Ministry of Culture and Communication, Regional Archaeology Service": "government/administrative body",

    # Archaeology/epigraphic RESEARCH institutes mismatched as museums (the "archaeolog"/"epigraphic"
    # keywords don't distinguish a museum's archaeological collection from a research institute)
    "German Archaeological Institute": "academic/research institution",
    "Epigraphic Database Roma": "academic/research institution",
    "Epigraphic Dabatase Bari": "academic/research institution",  # typo "Dabatase" is in the raw provider name
    "University Institute for Research in Iberian Archeology": "academic/research institution",
    "CISA -Interdipartimental Center for Archaeology": "academic/research institution",

    # "Conservatory" false-friend matches -- not music institutions despite the word
    "National Conservatory of Arts and Crafts": "academic/research institution",  # CNAM, a French engineering university
    "Conservatory of the Gironde Estuary": "government/administrative body",  # environmental conservation body

    # Science/technology museums swept into the generic art/history museum bucket
    "Museon-Omniversum": "natural history/science institution",
    "Technoseum": "natural history/science institution",
    "Zoological Research Museum Koenig": "natural history/science institution",
}


## 8. Review `other`/`None`, manual override

In [17]:
review_rows = []
for name in distinct_providers["provider"]:
    if name in CONFIRMED_FIXES:
        continue  # already resolved in code, no need to review again

    category, source = classify_provider(name)
    if category in (None, "other"):
        wd = wikidata_resolution.get(name, {})
        item_count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
        review_rows.append({
            "provider": name,
            "description_en": wd.get("description_en"),
            "category": category,
            "source": source,
            "count": item_count,
        })

new_review_df = pd.DataFrame(review_rows).sort_values("count", ascending=False)

review_path = DATA_DIR / "manual_review.csv"
if review_path.exists():
    existing_df = pd.read_csv(review_path)
    existing_categories = dict(zip(existing_df["provider"], existing_df["manual_category"]))
    new_review_df["manual_category"] = new_review_df["provider"].map(existing_categories).fillna("")
else:
    new_review_df["manual_category"] = ""

new_review_df.to_csv(review_path, index=False)
print(f"{len(new_review_df)} providers need review, {new_review_df['count'].sum():,} total items")
filled = (new_review_df["manual_category"] != "").sum()
print(f"{filled} already have a manual_category from a previous pass")
new_review_df.head(30)

77 providers need review, 1,834,363 total items
0 already have a manual_category from a previous pass


,provider,description_en,category,source,count,manual_category
27,German Documentation Center for Art History - ...,None,other,unmatched,320532,
39,Digital Memory of Catalonia,None,other,unmatched,286512,
40,Maresía: Prensa digitalizada y Patrimonio docu...,None,other,unmatched,141919,
45,Historical Information Center of Brabant,None,other,unmatched,118373,
0,Internet Culturale,Istituto Centrale per il Catalogo Unico web po...,other,unmatched,109101,
41,Centro de Estudios de Castilla - La Mancha,None,other,unmatched,94223,
46,Association De Hollandsche Molen,None,other,unmatched,94187,
1,Central Institute for the Union Catalogue of I...,Italian government agency,other,name_description_conflict,66672,
47,Historic Center Limburg,None,other,unmatched,62704,
48,Historic Center Leeuwarden,None,other,unmatched,53417,


**Review `data/manual_review.csv`**, sorted by `count` descending, save, then reload:


In [18]:
review_df = pd.read_csv(DATA_DIR / "manual_review.csv")
review_df["manual_category"] = review_df["manual_category"].fillna("")
manual_overrides = {
    row["provider"]: row["manual_category"].strip()
    for _, row in review_df.iterrows()
    if row["manual_category"].strip()
}
print(f"{len(manual_overrides)} manual overrides loaded from CSV")
print(f"{len(CONFIRMED_FIXES)} confirmed fixes hard-coded")


def classify_provider_final(name):
    if name in CONFIRMED_FIXES:
        return CONFIRMED_FIXES[name], "confirmed_fix"
    if name in manual_overrides:
        return manual_overrides[name], "manual_override"
    return classify_provider(name)


77 manual overrides loaded from CSV
15 confirmed fixes hard-coded


## 9. Aggregate by country, weighted by item volume

In [19]:
final_categories = {name: classify_provider_final(name)[0] for name in distinct_providers["provider"]}

top_providers_df["provider_category"] = top_providers_df["provider"].map(final_categories)
top_providers_df["provider_category"] = top_providers_df["provider_category"].fillna("unresolved")

category_by_country = (
    top_providers_df.groupby(["country", "provider_category"])["count"]
    .sum()
    .unstack(fill_value=0)
)

# bring in the TRUE total per country from step 1 (not just the sum of sampled providers)
category_by_country = category_by_country.merge(
    totals_df.set_index("country")["total_items"], left_index=True, right_index=True
)

sample_total = category_by_country.drop(columns="total_items").sum(axis=1)

# share of each country's TRUE total (will sum to LESS than 100% -- the gap is the unsampled long tail)
category_share = category_by_country.drop(columns="total_items").div(
    category_by_country["total_items"], axis=0
)

# how much of the true total the sample actually covers, per country
category_share["coverage"] = sample_total / category_by_country["total_items"]

category_share = (category_share * 100).round(2)
category_share = category_share.reset_index()

category_share.to_csv(DATA_DIR / "complete_df.csv", index=False)
category_share

,country,academic/research institution,art/history museum,audiovisual/film archive,government/administrative body,library/archive,media/broadcast organization,natural history/science institution,other,coverage
0,france,0.38,1.76,1.86,8.77,76.58,0.00,10.63,0.02,100.00
1,germany,2.28,13.71,0.92,0.00,76.03,0.64,4.04,0.13,97.74
2,italy,13.43,7.09,31.64,0.14,46.19,0.00,0.10,1.31,99.90
3,netherlands,2.85,9.03,1.76,6.61,26.93,0.00,51.00,1.82,100.00
4,portugal,5.30,4.73,0.47,4.54,37.53,0.36,46.95,0.13,100.00
5,spain,7.00,5.03,0.00,5.28,78.29,1.35,1.42,0.00,98.37


### 9.1 Full per-provider audit

Recomputes directly from `classify_provider_final` every time this runs -- does not depend on
`top_providers_df["provider_category"]` being freshly set by another cell, so it can't go
stale from running cells out of order (the cause of an earlier bug where confirmed manual
fixes weren't reflected in this export).


In [23]:
audit_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider_final(name)
    count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
    country = distinct_providers.loc[distinct_providers["provider"] == name, "country"].iloc[0]
    audit_rows.append({
        "provider": name,
        "country": country,
        "count": count,
        "provider_category": category if category else "unresolved",
        "classification_source": source if source else "unresolved",
    })

cat_df = pd.DataFrame(audit_rows).sort_values(["country", "count"], ascending=[True, False])
cat_df.to_csv(DATA_DIR / "cat_full.csv", index=False)

pd.set_option("display.max_rows", None)
cat_df.head(30)


,provider,country,count,provider_category,classification_source
100,National Library of France,france,2999310,library/archive,provider_name
101,Media Library of Architecture and Heritage,france,510877,library/archive,provider_name
102,Natural History Museum in Paris,france,502160,natural history/science institution,provider_name
103,Ministry of Culture,france,173279,government/administrative body,provider_name
104,Historical Monuments: Regional Conservation,france,128192,government/administrative body,confirmed_fix
105,"Ministry of Culture and Communication, Regiona...",france,110142,government/administrative body,confirmed_fix
106,National Audiovisual Institute France,france,54491,audiovisual/film archive,provider_name
107,Interuniversity Health Library,france,47992,library/archive,provider_name
108,Palais Galliera - Musée de la Mode de la Ville...,france,44495,art/history museum,provider_name
109,Mobilier National Collections,france,24284,art/history museum,manual_override


## 10. Visualization -- sunburst, country contribution -> provider typology

**Level 1 (inner ring):** each country's share of item volume, among the six countries
studied here (not a global Europeana share -- only these six countries and their top-100
sampled providers each are represented).

**Level 2 (outer ring, click a country to drill in):** that country's provider-typology
composition -- the same category breakdown as before, now explorable per country instead of
compared all at once in a single bar chart.


In [22]:
COUNTRY_COLORS = {
    "netherlands": "#a180ad",
    "france": "#1f7f95",
    "portugal": "#f4a64e",
    "italy": "#90BE6D",
    "germany": "#feda15",
    "spain": "#bb521f",
}

CATEGORY_COLORS = {
    "audiovisual/film archive": "#C2528C",              # rose/magenta
    "art/history museum": "#5C6BC0",                     # indigo
    "natural history/science institution": "#2A9D8F",   # teal-green
    "library/archive": "#6FA8DC",                        # soft sky blue
    "academic/research institution": "#7E6B8F",          # dusty violet-gray
    "media/broadcast organization": "#D4A24C",           # muted gold (deliberately duller than Portugal's brighter orange)
    "government/administrative body": "#8C4A4A",         # muted brick red
    "other": "#B9B9B9",                                  # neutral gray
    "unresolved": "#7A7A7A",                             # darker neutral gray
}
provider_counts_by_country = distinct_providers.groupby("country").size()
total_providers = len(distinct_providers)

sunburst_country = cat_df.groupby("country")["count"].sum().reset_index()
sunburst_cat = cat_df.groupby(["country", "provider_category"])["count"].sum().reset_index()

ids, labels, parents, values, colors, hover_text = [], [], [], [], [], []

# level 1 -- countries, colored individually, hover shows % of PROVIDERS (not % of items)
for _, row in sunburst_country.iterrows():
    country = row["country"]
    item_count = row["count"]
    n_providers = provider_counts_by_country.get(country, 0)
    provider_pct = n_providers / total_providers * 100

    ids.append(country)
    labels.append(country.capitalize())
    parents.append("")
    values.append(item_count)
    colors.append(COUNTRY_COLORS.get(country, "#CCCCCC"))
    hover_text.append(
        f"<b>{country.capitalize()}</b><br>{item_count:,} items<br>{provider_pct:.1f}% of all providers"
    )

# level 2 -- categories within each country, hover shows % of THAT COUNTRY's items
for _, row in sunburst_cat.iterrows():
    country, cat, count = row["country"], row["provider_category"], row["count"]
    country_total = sunburst_country.loc[sunburst_country["country"] == country, "count"].iloc[0]
    cat_pct_of_country = count / country_total * 100

    ids.append(f"{country}-{cat}")
    labels.append(cat)
    parents.append(country)
    values.append(count)
    colors.append(CATEGORY_COLORS.get(cat, "#D1D5DB"))
    hover_text.append(
        f"<b>{cat}</b><br>{count:,} items<br>{cat_pct_of_country:.1f}% of {country.capitalize()}"
    )

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    marker=dict(colors=colors),
    customdata=hover_text,
    hovertemplate="%{customdata}<extra></extra>",
    insidetextorientation="horizontal",
))

fig.update_layout(height=750, margin=dict(t=30, l=0, r=0, b=0))
fig.show()